# nb_analise_visual_gold — Visualizações Premium · Dados Públicos

**Objetivo:** Protótipo de alta fidelidade com Plotly replicando os painéis do portal Osasco.  
**Fonte:** Gold tables `lh_dados_publicos` via SQL Analytics Endpoint.  
**Clusters:** Santos · Osasco · Mauá (15 municípios)

In [ ]:
import pyodbc
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

SERVER   = 'ena6obg6j2cevcppw7dn7yu57a-knnp5frchjbujdik4l3nmgrgsa.datawarehouse.fabric.microsoft.com'
DATABASE = 'lh_dados_publicos'

def get_conn():
    return pyodbc.connect(
        f'Driver={{ODBC Driver 18 for SQL Server}};'
        f'Server={SERVER};Database={DATABASE};'
        'Encrypt=yes;TrustServerCertificate=no;Authentication=ActiveDirectoryInteractive'
    )

def q(sql):
    with get_conn() as conn:
        return pd.read_sql(sql, conn)

# Paleta de cores por cluster
CLUSTER_COLOR = {
    'SANTOS': '#1f77b4',
    'OSASCO': '#ff7f0e',
    'MAUA':   '#2ca02c',
}

TEMPLATE = 'plotly_white'
print('[OK] Setup concluído — Plotly pronto')

---
## Eixo Demográfico
### Gráfico 1 — Pirâmide Etária (Plotly · WOW factor)

Replicação do painel do portal Osasco com visão comparativa por município.  
Altere `MUNICIPIO_ALVO` para qualquer um dos 15 municípios.

In [ ]:
MUNICIPIO_ALVO = 'Santos'  # <-- altere aqui

df_pir = q(f"""
    SELECT idade, idade_codigo, sexo,
           SUM(TRY_CAST(valor AS FLOAT)) AS populacao
    FROM gold_censo_piramide_populacao
    WHERE municipio LIKE '%{MUNICIPIO_ALVO}%'
      AND ano = '2022'
      AND sexo IN ('Homens', 'Mulheres')
      AND forma_de_declaracao_da_idade = 'Total'
    GROUP BY idade, idade_codigo, sexo
    ORDER BY idade_codigo
""")

homens   = df_pir[df_pir['sexo'] == 'Homens'].sort_values('idade_codigo')
mulheres = df_pir[df_pir['sexo'] == 'Mulheres'].sort_values('idade_codigo')
faixas   = homens['idade'].tolist()

fig = go.Figure()
fig.add_trace(go.Bar(
    y=faixas, x=[-v for v in homens['populacao']],
    orientation='h', name='Homens',
    marker_color='#4878CF', text=homens['populacao'].apply(lambda v: f'{v:,.0f}'),
    textposition='outside', textfont_size=9,
))
fig.add_trace(go.Bar(
    y=faixas, x=mulheres['populacao'],
    orientation='h', name='Mulheres',
    marker_color='#D65F5F', text=mulheres['populacao'].apply(lambda v: f'{v:,.0f}'),
    textposition='outside', textfont_size=9,
))

max_v = max(homens['populacao'].max(), mulheres['populacao'].max()) * 1.15
tickvals = list(range(-int(max_v), int(max_v)+1, int(max_v/4)))

fig.update_layout(
    title=dict(text=f'<b>Pirâmide Etária — {MUNICIPIO_ALVO}</b><br><sup>Censo 2022 · IBGE SIDRA t/9514</sup>', x=0.5),
    barmode='overlay',
    xaxis=dict(
        tickvals=tickvals,
        ticktext=[f'{abs(v):,}' for v in tickvals],
        title='População',
    ),
    yaxis_title='Faixa Etária',
    template=TEMPLATE,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=550, width=820,
    bargap=0.1,
)
fig.add_vline(x=0, line_width=1, line_color='black')
fig.show()

### Gráfico 2 — Envelhecimento Comparativo (2010 vs 2022)

Índice de Envelhecimento = (pop 65+ / pop 0–14) × 100. Valores > 100 indicam mais idosos que jovens.

In [ ]:
df_env = q("""
    SELECT municipio, ano, TRY_CAST(valor AS FLOAT) AS indice
    FROM gold_censo_envelhecimento
    WHERE cor_ou_raca = 'Total'
    ORDER BY ano, municipio
""")
df_env['municipio'] = df_env['municipio'].str.replace(r'\s*\(SP\)', '', regex=True)
df_env['ano'] = df_env['ano'].astype(str)

fig = px.bar(
    df_env, x='municipio', y='indice', color='ano', barmode='group',
    color_discrete_sequence=['#9ecae1', '#2171b5'],
    labels={'indice': 'Índice de Envelhecimento', 'municipio': '', 'ano': 'Censo'},
    title='<b>Índice de Envelhecimento por Município</b><br><sup>Razão pop 65+ / pop 0–14 × 100 · Comparativo Censos</sup>',
    template=TEMPLATE, height=480,
)
fig.add_hline(y=100, line_dash='dash', line_color='red',
              annotation_text='Equilíbrio (100)', annotation_position='top right')
fig.update_layout(xaxis_tickangle=-35, title_x=0.5)
fig.show()

### Gráfico 3 — Dispersão: Formalização × Envelhecimento

Cruzamento entre o mercado de trabalho (RAIS) e a estrutura demográfica.  
Permite identificar se municípios mais envelhecidos têm maior ou menor formalização.

In [ ]:
# Formalização (RAIS 2022 / populacao)
df_formal = q("""
    WITH rais AS (
        SELECT id_municipio, nome_municipio, cluster,
               SUM(CAST(vinculos_ativos AS FLOAT)) AS vinculos
        FROM gold_mercado_trabalho
        WHERE fonte = 'RAIS' AND ano = 2022
        GROUP BY id_municipio, nome_municipio, cluster
    ),
    pop AS (
        SELECT id_municipio, SUM(TRY_CAST(populacao AS FLOAT)) AS populacao
        FROM gold_populacao_municipios
        WHERE ano = (SELECT MAX(ano) FROM gold_populacao_municipios)
        GROUP BY id_municipio
    )
    SELECT r.nome_municipio, r.cluster,
           ROUND((r.vinculos / NULLIF(p.populacao,0)) * 100, 1) AS pct_formalizacao
    FROM rais r
    JOIN pop p ON CAST(r.id_municipio AS BIGINT) = CAST(p.id_municipio AS BIGINT)
""")

# Envelhecimento 2022
df_env2 = q("""
    SELECT municipio, TRY_CAST(valor AS FLOAT) AS indice_env
    FROM gold_censo_envelhecimento
    WHERE cor_ou_raca = 'Total' AND ano = '2022'
""")
df_env2['municipio'] = df_env2['municipio'].str.replace(r'\s*\(SP\)', '', regex=True)

df_scatter = df_formal.merge(df_env2, left_on='nome_municipio', right_on='municipio', how='inner')

fig = px.scatter(
    df_scatter, x='indice_env', y='pct_formalizacao',
    color='cluster', text='nome_municipio',
    color_discrete_map=CLUSTER_COLOR,
    size=[15]*len(df_scatter),
    labels={
        'indice_env': 'Índice de Envelhecimento (2022)',
        'pct_formalizacao': '% Formalização (RAIS / Pop)',
        'cluster': 'Cluster',
    },
    title='<b>Formalização × Envelhecimento por Município</b><br><sup>Cada ponto = 1 município · Censo 2022 + RAIS 2022</sup>',
    template=TEMPLATE, height=500,
)
fig.update_traces(textposition='top center', textfont_size=10)
fig.update_layout(title_x=0.5)
fig.show()

---
## Eixo Econômico
### Gráfico 4 — Série Temporal de Empregos Formais por Cluster

Saldo CAGED (fluxo mensal) e estoque RAIS (anual) unificados em `gold_mercado_trabalho`.

In [ ]:
# RAIS: estoque anual de vínculos por cluster
df_rais = q("""
    SELECT cluster, ano,
           SUM(CAST(vinculos_ativos AS FLOAT)) AS vinculos
    FROM gold_mercado_trabalho
    WHERE fonte = 'RAIS'
    GROUP BY cluster, ano
    ORDER BY cluster, ano
""")

# CAGED: saldo acumulado anual por cluster
df_caged = q("""
    SELECT cluster, ano,
           SUM(CAST(saldo_mensal AS FLOAT)) AS saldo
    FROM gold_mercado_trabalho
    WHERE fonte = 'CAGED'
    GROUP BY cluster, ano
    ORDER BY cluster, ano
""")

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Estoque de Vínculos Ativos — RAIS', 'Saldo Líquido de Empregos — CAGED'),
    shared_yaxes=False,
)

for cluster in ['SANTOS', 'OSASCO', 'MAUA']:
    cor = CLUSTER_COLOR[cluster]
    grp_r = df_rais[df_rais['cluster'] == cluster]
    grp_c = df_caged[df_caged['cluster'] == cluster]

    fig.add_trace(go.Scatter(
        x=grp_r['ano'], y=grp_r['vinculos'], name=cluster,
        line=dict(color=cor, width=2), mode='lines+markers',
        legendgroup=cluster, showlegend=True,
    ), row=1, col=1)

    fig.add_trace(go.Bar(
        x=grp_c['ano'], y=grp_c['saldo'], name=cluster,
        marker_color=cor, legendgroup=cluster, showlegend=False,
    ), row=1, col=2)

fig.add_hline(y=0, line_dash='dot', line_color='black', row=1, col=2)
fig.update_layout(
    title=dict(text='<b>Mercado de Trabalho por Cluster</b><br><sup>gold_mercado_trabalho · RAIS + CAGED unificados</sup>', x=0.5),
    template=TEMPLATE, height=450, barmode='group',
    legend=dict(orientation='h', yanchor='bottom', y=1.05),
)
fig.show()

### Gráfico 5 — PIB por Município (Evolução + Composição)

Evolução do PIB total e composição por setor econômico (VAB) via `gold_pib_municipios`.

In [ ]:
# Inspecionar schema primeiro
cols_pib = list(pd.read_sql('SELECT TOP 0 * FROM gold_pib_municipios', get_conn()).columns)
print('Colunas gold_pib_municipios:', cols_pib)
df_pib_sample = q('SELECT TOP 3 * FROM gold_pib_municipios')
print(df_pib_sample.to_string())

In [ ]:
# AJUSTE A QUERY ABAIXO COM OS NOMES REAIS DAS COLUNAS APÓS RODAR A CÉLULA ACIMA
# Exemplo assumindo colunas: nome_municipio, cluster, ano, pib_total, pib_per_capita

df_pib = q("""
    SELECT nome_municipio, cluster, ano,
           TRY_CAST(pib_total AS FLOAT) AS pib_total,
           TRY_CAST(pib_per_capita AS FLOAT) AS pib_per_capita
    FROM gold_pib_municipios
    ORDER BY ano, nome_municipio
""")

# PIB per capita por município — último ano disponível
ano_max = df_pib['ano'].max()
df_ult = df_pib[df_pib['ano'] == ano_max].sort_values('pib_per_capita', ascending=True)

fig = px.bar(
    df_ult, x='pib_per_capita', y='nome_municipio', color='cluster',
    color_discrete_map=CLUSTER_COLOR, orientation='h',
    text='pib_per_capita',
    labels={'pib_per_capita': 'PIB per Capita (R$ mil)', 'nome_municipio': ''},
    title=f'<b>PIB per Capita por Município — {ano_max}</b><br><sup>gold_pib_municipios · IBGE SIDRA</sup>',
    template=TEMPLATE, height=500,
)
fig.update_traces(texttemplate='R$ %{text:,.1f}k', textposition='outside')
fig.update_layout(title_x=0.5, showlegend=True)
fig.show()

---
## Eixo Segurança Pública — Placeholder

> **Status:** Aguardando ingestão SSP-SP (`ctrl_ssp_releases` configurado).  
> Quando `gold_seguranca_publica` estiver disponível, implementar aqui:
> - Série temporal de ocorrências por tipo de delito
> - Mapa de calor por município × categoria
> - Comparativo CVLI (Crimes Violentos Letais Intencionais) por cluster

---
## Verificação de Paridade — Fabric vs Portal Osasco

Comparar valores gerados aqui com os visíveis no portal para confirmar que as tabelas Gold são corretas.

In [ ]:
# Snapshot dos valores-chave para comparar com o portal
print('=== PARIDADE DE DADOS — Fabric vs Portal Osasco ===')

osasco_id = 3534401

# Pop total Osasco
pop_osasco = q(f"""
    SELECT ano, populacao FROM gold_populacao_municipios
    WHERE id_municipio = {osasco_id}
    ORDER BY ano DESC
""").head(3)
print('\nPopulação Osasco (últimos 3 anos):')
print(pop_osasco.to_string(index=False))

# Saldo CAGED Osasco 2024
caged_osasco = q(f"""
    SELECT ano, SUM(CAST(saldo_mensal AS FLOAT)) AS saldo_anual
    FROM gold_mercado_trabalho
    WHERE id_municipio = {osasco_id} AND fonte = 'CAGED' AND ano = 2024
    GROUP BY ano
""")
print('\nSaldo CAGED Osasco 2024:')
print(caged_osasco.to_string(index=False))

# Envelhecimento Osasco
env_osasco = q("""
    SELECT ano, valor AS indice_envelhecimento
    FROM gold_censo_envelhecimento
    WHERE municipio LIKE '%Osasco%' AND cor_ou_raca = 'Total'
    ORDER BY ano
""")
print('\nÍndice de Envelhecimento Osasco:')
print(env_osasco.to_string(index=False))